# Assignment 1 — Data Cleaning
### Retail Sales Dataset  |  100 Marks  |  8 Tasks

---

**Dataset:** `retail_sales_dataset.csv`  —  4,310 rows x 21 columns

| Assignment | Notebook |
|---|---|
| Assignment 1 | Data Cleaning |
| Assignment 2 | Exploratory Data Analysis |
| Assignment 3 | Statistical Analysis |
| Assignment 4 | Segmentation and Clustering |
| Assignment 5 | Time Series Forecasting |

> **Note:** This notebook covers Assignment 1 only. Run cells top-to-bottom for best results.


## Environment Setup

In [ ]:
# Run once to install any missing packagesimport subprocess, syspkgs = ['pandas','numpy','matplotlib','seaborn','scipy','statsmodels','scikit-learn','pmdarima','prophet']for p in pkgs:subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)print("All packages ready.")

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsimport warningswarnings.filterwarnings('ignore')# Plot stylesns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(12,5),'axes.titlesize':13, 'axes.labelsize':11})print("Imports complete ")

# Assignment 1 — Data Cleaning
**Marks: 100  |  Tasks: 8**

---

## Why Data Cleaning Matters

Real-world datasets are never perfect. Before any analysis or machine learning can begin, raw data must be systematically cleaned and validated. Data cleaning (also called data wrangling or data munging) is the process of detecting and correcting corrupt, inaccurate, incomplete, or irrelevant records from a dataset.

Studies consistently show that data scientists spend 60–80% of their time on data preparation tasks — making data cleaning one of the most critical and time-consuming skills in the field.

### What This Assignment Covers

In this assignment you will work through the complete data cleaning lifecycle on a retail sales dataset that contains deliberately injected data quality issues:

- **Missing values** (~3–8% per column, injected randomly and systematically)
- **Exact duplicate rows** (~80 rows duplicated)
- **Blank rows** (~30 rows completely null)
- **Inconsistent text casing** in gender and order_status columns
- **Mixed date formats** in the order_date column (three different formats)
- **Numeric outliers** in age, quantity, days_to_ship, and revenue
- **Wrong data types** (numeric columns stored as strings)

### Cleaning Workflow (8 Tasks)

| Task | Topic | Key Technique |
|---|---|---|
| 1 | Initial Audit | isnull, duplicated, value_counts |
| 2 | Handle Duplicates | drop_duplicates, near-duplicate logic |
| 3 | Handle Missing Values | median/mode/group-wise imputation |
| 4 | Standardize Categorical Columns | str.strip, str.title, str.lower |
| 5 | Parse and Fix Date Column | dateutil, pd.to_datetime, dt accessor |
| 6 | Outlier Detection and Treatment | IQR, Z-score, Winsorize, clip |
| 7 | Data Type Correction | astype, pd.to_numeric, pd.Categorical |
| 8 | Final Validation and Export | assert statements, to_csv |

### Learning Objectives

By the end of this assignment you will be able to:
1. Perform a systematic data quality audit on any tabular dataset
2. Choose the correct imputation strategy based on data distribution and missingness mechanism
3. Detect and handle outliers using multiple statistical methods
4. Parse and standardize heterogeneous date formats
5. Build a reusable data validation function
6. Produce a clean, analysis-ready dataset with documented cleaning decisions


---
## Task 1 — Initial Audit

### Theory: What is a Data Audit?

Before modifying any data, you must understand its current state. A data audit is a structured inspection that produces a baseline profile of every column. It answers four fundamental questions:

1. **Completeness** — Are there missing values? In which columns and how many?
2. **Uniqueness** — Are there duplicate rows?
3. **Consistency** — Do categorical values use consistent formatting?
4. **Validity** — Are values within plausible ranges for their domain?

### Key Methods

| Method | What it Shows |
|---|---|
| `df.shape` | Total rows and columns |
| `df.dtypes` | Data type of each column |
| `df.isnull().sum()` | Count of NaN/None per column |
| `df.duplicated().sum()` | Count of exact duplicate rows |
| `df.describe()` | Min, max, mean, std, quartiles for numeric columns |
| `df.value_counts()` | Frequency of each unique value in a categorical column |

### Missing Value Mechanisms (Important for Imputation)

Understanding *why* data is missing guides *how* to fill it:

- **MCAR (Missing Completely At Random):** The probability of a value being missing is the same for all observations. Safe to impute or drop.
- **MAR (Missing At Random):** Missingness depends on other observed variables but not on the missing value itself. Impute carefully.
- **MNAR (Missing Not At Random):** Missingness depends on the unobserved value (e.g., high-income people not reporting income). Imputing is risky; flagging is safer.

### Practical Thresholds

- **0–5% missing:** Impute with median/mode — minimal risk
- **5–20% missing:** Use group-wise or model-based imputation
- **20–50% missing:** Consider multiple imputation; evaluate column usefulness
- **>50% missing:** Usually drop the column unless it carries unique signal


In [ ]:
# Load the raw datasetdf_raw = pd.read_csv('retail_sales_dataset.csv')print("=" * 55)print(f" Shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")print("=" * 55)print(f"\nColumn names:\n {list(df_raw.columns)}")print(f"\nData types:")print(df_raw.dtypes.to_string())

In [ ]:
# Missing value auditmissing_count = df_raw.isnull().sum()missing_pct = (missing_count / len(df_raw) * 100).round(2)audit = pd.DataFrame({'Missing Count': missing_count,'Missing %': missing_pct,'Dtype': df_raw.dtypes}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)print("Columns with missing values:")print(audit.to_string())print(f"\nTotal missing cells: {df_raw.isnull().sum().sum():,}")print(f"Rows with at least 1 null: {df_raw.isnull().any(axis=1).sum():,}")

In [ ]:
# Missing value heatmap# Yellow stripe = missing cell; dark = presentfig, axes = plt.subplots(1, 2, figsize=(15, 5))# Heatmapsns.heatmap(df_raw.isnull(), cbar=False, yticklabels=False,cmap='YlOrRd', ax=axes[0])axes[0].set_title('Missing Values Heatmap\n(red stripe = missing cell)', fontsize=12)axes[0].set_xlabel('Columns')# Bar chart of missing %cols_with_miss = missing_pct[missing_pct > 0].sort_values(ascending=True)cols_with_miss.plot(kind='barh', color='steelblue', ax=axes[1])axes[1].axvline(5, color='red', linestyle='--', label='5% threshold')axes[1].set_xlabel('Missing %')axes[1].set_title('Missing % per Column')axes[1].legend()plt.tight_layout()plt.show()

In [ ]:
# Duplicate detectionn_exact_dupes = df_raw.duplicated().sum()print(f"Exact duplicate rows: {n_exact_dupes}")# Show 3 duplicate pairsdup_mask = df_raw.duplicated(keep=False)dupes = df_raw[dup_mask].sort_values(list(df_raw.columns[:4]))print("\nSample duplicate rows (first 6 shown):")print(dupes.head(6)[['order_id','order_date','customer_id','product_name','sales_amount']].to_string())

---
## Task 2 — Handle Duplicates

### Theory: Types of Duplicates

**Exact duplicates** are rows where every single column value is identical. These arise from system errors, double-entry, or accidental data merges. They inflate counts, distort means, and introduce bias in any downstream model.

**Near-duplicates** share the same business key (e.g., same customer + same order date + same product) but differ in one or two non-key columns. These are harder to detect and require domain knowledge to resolve.

### Why Duplicates Are Harmful

Consider a retail dataset where 80 orders are duplicated:
- Total revenue appears 80 orders higher than reality
- Customer purchase frequency is overstated
- Any model trained on this data learns from phantom transactions

### Detection Strategy

```python
# Step 1: Exact duplicates
df.duplicated().sum()

# Step 2: Key-based duplicates (same order_id but different fields)
df.duplicated(subset=['order_id'], keep=False)

# Step 3: Near-duplicates (same customer, product, date — different revenue)
df.duplicated(subset=['customer_id', 'product_name', 'order_date'], keep=False)
```

### Resolution Strategy

| Duplicate Type | Resolution |
|---|---|
| Exact duplicate | Keep first occurrence (`drop_duplicates(keep='first')`) |
| Key-based duplicate | Investigate; keep the record with the most complete data |
| Near-duplicate (value differs) | Escalate to data owner or average the conflicting values |

### Best Practice

Always document:
- How many rows were dropped
- Which deduplication rule was applied
- Whether the dropped rows showed any pattern (e.g., concentrated in one date range)


In [ ]:
# Working copydf = df_raw.copy()shape_before = df.shape# Drop exact duplicates (keep first occurrence)df = df.drop_duplicates(keep='first')print(f"Rows before: {shape_before[0]:,}")print(f"Rows after: {df.shape[0]:,}")print(f"Exact duplicates dropped: {shape_before[0] - df.shape[0]}")

In [ ]:
# Near-duplicate detection# Near-dup = same customer_id + order_date + product_name but different order_idnear_dup_cols = ['customer_id', 'order_date', 'product_name']near_dupes = df[df.duplicated(subset=near_dup_cols, keep=False)]print(f"Near-duplicate groups (same customer+date+product): {len(near_dupes)}")if len(near_dupes) > 0:print("\nSample near-duplicates:")print(near_dupes[near_dup_cols + ['order_id','sales_amount']].head(8).to_string())# Strategy: keep the FIRST occurrence (earliest order_id sorts first)# Rationale: duplicate entries are most likely data pipeline re-ingestion errorsdf = df.drop_duplicates(subset=near_dup_cols, keep='first')print(f"\nRows after near-dup removal: {df.shape[0]:,}")

---
## Task 3 — Handle Missing Values

### Theory: The Imputation Decision Framework

There is no single correct way to handle missing values. The right approach depends on three factors:

1. **The missingness mechanism** (MCAR, MAR, or MNAR — see Task 1)
2. **The variable's distribution** (symmetric vs skewed, categorical vs numeric)
3. **The downstream use** (EDA vs machine learning model)

### Imputation Methods Compared

| Method | Formula | Best For | Weakness |
|---|---|---|---|
| Mean imputation | x_hat = mean(x) | Symmetric, low-missing numeric | Distorts skewed distributions, reduces variance |
| Median imputation | x_hat = median(x) | Skewed numeric, outlier-prone | Slightly less efficient than mean for normal data |
| Mode imputation | x_hat = most_frequent(x) | Categorical and discrete variables | Introduces bias toward dominant category |
| Constant fill | x_hat = 'Unknown' or 0 | When missingness is meaningful | Hides the signal in missingness |
| Group-wise median | x_hat = median(x | group) | When distribution varies by group | Requires a meaningful grouping variable |
| KNN imputation | x_hat = weighted mean of k nearest | Correlated numeric features | Slow for large datasets |
| Forward fill (ffill) | x_hat = previous observed value | Time-ordered data | Wrong for unordered data |

### Variance Reduction from Imputation

Imputing with the mean (or median) always reduces the observed variance of the column. This is acceptable if the imputed column will only be used for grouping or filtering, but problematic for regression where standard errors matter.

**Rule of thumb:** For any column used as a dependent variable (target), never impute — either remove rows with missing targets or use multiple imputation.

### Flagging Strategy

When you impute a value, create a binary flag column recording which rows were imputed:

```python
df['age_was_missing'] = df['age'].isnull().astype(int)
df['age'].fillna(df['age'].median(), inplace=True)
```

This preserves the information that missingness itself may carry for downstream models.


In [ ]:
# Drop fully blank rows (completely null rows injected into dataset)n_all_null = df.isnull().all(axis=1).sum()df = df.dropna(how='all')print(f"Fully blank rows removed: {n_all_null}")# Drop rows where structural keys are missingcritical = ['order_date', 'order_status', 'region']before = df.shape[0]df = df.dropna(subset=critical)print(f"Rows dropped (missing critical columns): {before - df.shape[0]}")print(f"Remaining rows: {df.shape[0]:,}")

In [ ]:
# age: flag then fill with mediandf['age_missing'] = df['age'].isnull().astype(int)age_median = df['age'].median()df['age'] = df['age'].fillna(age_median)print(f"age: {df['age_missing'].sum()} rows were missing → filled with median ({age_median:.1f})")print("Why median? The column has injected outliers (0, 150, 999) that distort the mean.")print(f" Mean before cleaning: {df_raw['age'].mean():.1f} vs Median: {age_median:.1f}")

In [ ]:
# customer_satisfaction: flag then fill with modedf['satisfaction_missing'] = df['customer_satisfaction'].isnull().astype(int)sat_mode = int(df['customer_satisfaction'].mode()[0])df['customer_satisfaction'] = df['customer_satisfaction'].fillna(sat_mode)print(f"customer_satisfaction: {df['satisfaction_missing'].sum()} missing → filled with mode ({sat_mode})")print("Why mode? It is an ordinal 1-5 scale. Mode = most common rating.")

In [ ]:
# discount_pct: fill with 0 (business logic)n_disc_miss = df['discount_pct'].isnull().sum()df['discount_pct'] = df['discount_pct'].fillna(0)print(f"discount_pct: {n_disc_miss} missing → filled with 0")print("Business logic: a missing discount record means no discount was applied.")# quantity & days_to_ship: group-wise median by product_categoryfor col in ['quantity', 'days_to_ship']:n_miss = df[col].isnull().sum()group_median = df.groupby('product_category')[col].transform('median')df[col] = df[col].fillna(group_median)# fallback: any remaining nulls (categories with all-null) use global mediandf[col] = df[col].fillna(df[col].median())print(f"{col}: {n_miss} missing → filled with group-wise median by product_category")

---
## Task 4 — Standardize Categorical Columns

### Theory: Why Categorical Consistency Matters

Pandas treats 'Male', 'male', 'MALE', and ' Male' as four completely different categories. Any groupby, pivot, or join operation will silently produce wrong results when categories are inconsistently named.

**Example of the problem:**

```
df['gender'].value_counts()
# Output:
# Male       1823
# male        412     ← same thing, treated as different
# MALE         89
# M            34
# Female      1690
# female       371
# F             28
```

This would produce 6 groups instead of 2, inflating row counts and producing meaningless per-group statistics.

### Standardization Operations

| Operation | Pandas Method | Example |
|---|---|---|
| Remove leading/trailing spaces | `.str.strip()` | ' Male ' -> 'Male' |
| Convert to lowercase | `.str.lower()` | 'MALE' -> 'male' |
| Convert to Title Case | `.str.title()` | 'male' -> 'Male' |
| Replace specific values | `.str.replace()` or `.map()` | 'M' -> 'Male' |
| Regex-based cleanup | `.str.replace(pattern, repl, regex=True)` | Remove special chars |

### Systematic Approach

1. Run `value_counts()` to see all unique values and their frequencies
2. Identify groups that represent the same category
3. Build a mapping dictionary for ambiguous codes
4. Apply `str.strip().str.title()` as the baseline standardization
5. Apply `map()` or `replace()` for remaining aliases

### Categorical Dtype Conversion

After standardizing text values, convert columns with limited unique values to pandas `Categorical` dtype:

```python
df['gender'] = df['gender'].astype('category')
```

Benefits:
- **Memory:** Stores codes (integers) instead of repeated strings — up to 90% memory reduction
- **Speed:** Groupby and sort operations are faster on Categorical
- **Validation:** Attempting to assign an unlisted category raises an error, catching future data issues early


In [ ]:
# gender: standardize all variantsprint("BEFORE — gender value counts:")print(df['gender'].value_counts().to_string())gender_map = {'M':'Male', 'male':'Male', 'MALE':'Male', 'm':'Male','F':'Female', 'female':'Female', 'FEMALE':'Female', 'f':'Female',}df['gender'] = df['gender'].replace(gender_map)print("\nAFTER — gender value counts:")print(df['gender'].value_counts().to_string())

In [ ]:
# order_status: strip whitespace, apply Title Caseprint("BEFORE — order_status unique:", df['order_status'].unique())df['order_status'] = df['order_status'].str.strip().str.title()print("AFTER — order_status unique:", df['order_status'].unique())# region: validate only expected values existvalid_regions = {'North','South','East','West','Central'}invalid_mask = ~df['region'].isin(valid_regions) & df['region'].notna()print(f"\nInvalid region values found: {invalid_mask.sum()}")print(df.loc[invalid_mask, 'region'].value_counts())if invalid_mask.sum() > 0:df = df[~invalid_mask]print("Rows with invalid regions dropped.")

---
## Task 5 — Parse and Fix the Date Column

### Theory: Why Date Parsing is Tricky

Date columns are among the most error-prone in real-world datasets. The order_date column in this dataset contains three different formats injected to simulate real production data quality issues:

- **ISO format:** `2023-01-15` (YYYY-MM-DD) — standard, unambiguous
- **European format:** `15/01/2023` (DD/MM/YYYY) — common in UK/EU systems
- **Long-form:** `January 15, 2023` — common in exports from CRM tools

If you apply `pd.to_datetime()` with a fixed format string, only one of these three formats will parse correctly — the other two will become NaT (Not a Time), silently losing data.

### Parsing Strategy

| Approach | When to Use | Risk |
|---|---|---|
| `pd.to_datetime(col)` with `infer_datetime_format=True` | Single uniform format | Misinterprets ambiguous dates (01/02 could be Jan 2 or Feb 1) |
| `pd.to_datetime(col, format='%Y-%m-%d')` | Known single format | Fails on any row with a different format |
| `dateutil.parser.parse()` via `.apply()` | Mixed formats | Slower; interprets ambiguous dates using US defaults |
| Custom try/except with multiple formats | Known multiple formats | Most explicit; safest |

### Ambiguity in Date Parsing

`dateutil.parser.parse('01/02/2023')` will return January 2 by default (US convention). To force day-first interpretation:

```python
from dateutil import parser
parser.parse('01/02/2023', dayfirst=True)  # -> February 1
```

Always validate after parsing: check that dates fall within the expected range for your dataset.

### Date Feature Engineering

Once the column is datetime64, the `.dt` accessor unlocks dozens of derived features:

| Feature | Code | Use Case |
|---|---|---|
| Year | `df['order_date'].dt.year` | Year-over-year trend analysis |
| Month | `df['order_date'].dt.month` | Seasonality analysis |
| Day of week | `df['order_date'].dt.dayofweek` | Weekday vs weekend patterns |
| Quarter | `df['order_date'].dt.quarter` | Quarterly reporting |
| Days to ship | `(ship_date - order_date).dt.days` | Logistics performance |
| Is weekend | `df['order_date'].dt.dayofweek >= 5` | Weekend effect on sales |

These derived features often carry more predictive signal than the raw date column itself.


In [ ]:
from dateutil import parser as dateutil_parserdef safe_parse(val):"""Parse a date string regardless of format. Returns NaT on failure."""if pd.isna(val):return pd.NaTtry:return pd.Timestamp(dateutil_parser.parse(str(val), dayfirst=False))except Exception:return pd.NaT# Parse — this may take ~5-10 seconds on 4k rowsdf['order_date'] = df['order_date'].apply(safe_parse)n_failed = df['order_date'].isna().sum()print(f"Parsing complete. Failed / unparseable: {n_failed}")print(f"dtype: {df['order_date'].dtype}")print(f"Range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")

In [ ]:
# Extract time featuresdf['year'] = df['order_date'].dt.yeardf['month'] = df['order_date'].dt.monthdf['quarter'] = df['order_date'].dt.quarterdf['day_of_week'] = df['order_date'].dt.dayofweek # 0=Mon, 6=Sundf['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)print("New time columns created:")print(df[['order_date','year','month','quarter','day_of_week','is_weekend']].head(5).to_string())# Validate date rangeout_of_range = df[(df['year'] < 2020) | (df['year'] > 2024)]print(f"\nDates outside 2020-2024: {len(out_of_range)}")df = df[(df['year'] >= 2020) & (df['year'] <= 2024)]print(f"Rows after date validation: {df.shape[0]:,}")

---
## Task 6 — Outlier Detection and Treatment

### Theory: What is an Outlier?

An outlier is an observation that lies an abnormal distance from other values in the dataset. Crucially, there are two types of outliers with very different implications:

1. **Error outliers:** Values that are wrong — e.g., age = 300, quantity = -50. These should be corrected or removed.
2. **Genuine extreme values:** Values that are correct but unusual — e.g., a single very large order worth 10x the average. These should be investigated, not blindly removed.

**Removing genuine outliers can destroy important business insights.** A single whale customer who spends 50x the average is not an error — they are your most valuable customer.

### Detection Methods

#### Method 1: IQR (Interquartile Range)
The most widely used method. Works for any distribution shape.

```
Q1 = 25th percentile
Q3 = 75th percentile
IQR = Q3 - Q1
Lower fence = Q1 - 1.5 * IQR
Upper fence = Q3 + 1.5 * IQR
```

Anything below the lower fence or above the upper fence is flagged as an outlier.

#### Method 2: Z-Score
Measures how many standard deviations a value is from the mean. Values with |Z| > 3 are flagged. **Only reliable for approximately normal distributions.**

```
Z = (x - mean) / std
Outlier if |Z| > 3
```

#### Method 3: Modified Z-Score (MAD)
Uses median and Median Absolute Deviation instead of mean and std — making it robust to the very outliers it is trying to detect.

```
MAD = median(|xi - median(x)|)
Modified Z = 0.6745 * (xi - median) / MAD
Outlier if |Modified Z| > 3.5
```

### Treatment Methods

| Treatment | Method | Best For |
|---|---|---|
| **Capping (Winsorize)** | Replace outliers with fence values | Retaining all rows; reducing influence |
| **Removal** | Drop outlier rows | Confirmed data entry errors |
| **Log transform** | `np.log1p(x)` | Right-skewed distributions with many small values and few large ones |
| **Binning** | Convert to ordinal bins | When exact numeric value matters less than the order |
| **Flag and keep** | Create `is_outlier` column | When the outlier status is itself informative for models |

### The Golden Rule

**Never remove outliers without investigation.** First visualize them. Then ask: is this value plausible? Does it follow a pattern? Does the analysis conclusion change materially when it is included or excluded?


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))for ax, col in zip(axes.flat, ['age','quantity','days_to_ship','sales_amount','profit','discount_pct']):df.boxplot(column=col, ax=ax, boxprops=dict(color='steelblue'),medianprops=dict(color='red',linewidth=2),flierprops=dict(marker='o',color='orange',markersize=4))ax.set_title(col, fontsize=11)ax.set_xlabel('')axes.flat[-1].set_visible(False) if len(axes.flat) > 5 else Noneplt.suptitle('Boxplots Before Outlier Treatment — orange dots are outliers', fontsize=13, y=1.01)plt.tight_layout()plt.show()

In [ ]:
# age: clip to valid domain [18, 90]age_before = df['age'].copy()df['age'] = df['age'].clip(lower=18, upper=90)clipped = (age_before != df['age']).sum()print(f"age: {clipped} values clipped to [18, 90]")print(f" Min before: {age_before.min():.0f} → After: {df['age'].min():.0f}")print(f" Max before: {age_before.max():.0f} → After: {df['age'].max():.0f}")# quantity: remove impossible valuesinvalid_qty = df[(df['quantity'] <= 0) | (df['quantity'] > 50)]print(f"\nquantity: {len(invalid_qty)} invalid rows (≤0 or >50) removed")df = df[(df['quantity'] > 0) & (df['quantity'] <= 50)]# days_to_ship: replace impossibles with NaN then re-imputeinvalid_ship = (df['days_to_ship'] < 1) | (df['days_to_ship'] > 30)print(f"days_to_ship: {invalid_ship.sum()} values out of [1,30] → replaced with group median")df.loc[invalid_ship, 'days_to_ship'] = np.nangroup_med = df.groupby('product_category')['days_to_ship'].transform('median')df['days_to_ship'] = df['days_to_ship'].fillna(group_med).fillna(df['days_to_ship'].median())

In [ ]:
from scipy.stats.mstats import winsorize# sales_amount: IQR outlier detection then Winsorizefor col in ['sales_amount']:Q1, Q3 = df[col].quantile([0.25, 0.75])IQR = Q3 - Q1lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQRn_out = ((df[col] < lo) | (df[col] > hi)).sum()print(f"{col}: {n_out} IQR outliers detected (bounds: {lo:,.0f} to {hi:,.0f})")df[col] = winsorize(df[col], limits=[0.01, 0.01])print(f" Winsorized at 1st/99th percentile → new range: {df[col].min():,.0f} to {df[col].max():,.0f}")# profit: show distribution (negatives are VALID — do not remove)n_neg_profit = (df['profit'] < 0).sum()print(f"\nprofit: {n_neg_profit} negative values — these are VALID loss-making orders.")print(" We DO NOT remove them. They are important for profitability analysis.")print(f" profit range: {df['profit'].min():,.0f} to {df['profit'].max():,.0f}")

---
## Task 7 — Data Type Correction

### Theory: Why Data Types Matter

When pandas reads a CSV file, it infers data types from the first few rows. If a numeric column contains even one non-numeric value (a currency symbol, a comma, a stray character), pandas assigns it `object` (string) dtype. This means:

- **Arithmetic fails silently:** `df['price'].mean()` raises a TypeError
- **Sorting is lexicographic:** '12' < '9' in string comparison
- **Memory is wasted:** Storing '123.45' as a string takes ~5x more memory than as float64
- **Statistical functions are unavailable:** `.describe()` shows string stats (count, unique, top, freq) instead of numeric stats

### Common Type Issues and Fixes

| Issue | Example | Fix |
|---|---|---|
| Numeric stored as string | '123.45' | `pd.to_numeric(col, errors='coerce')` |
| Currency-formatted number | '$1,234.56' | Strip `$` and `,` first, then to_numeric |
| Date stored as string | '2023-01-15' | `pd.to_datetime(col, errors='coerce')` |
| Boolean stored as string | 'True' / 'False' | `col.map({'True': True, 'False': False})` |
| Integer with nulls | `1.0, 2.0, NaN` | Pandas loads as float64 because int cannot hold NaN; use `pd.Int64Dtype()` |
| Category stored as object | 'Male', 'Female' | `.astype('category')` |

### The `errors='coerce'` Parameter

This is critical for safe type conversion. Without it, a single bad value crashes the entire conversion. With `errors='coerce'`, bad values become NaN, which you can then handle with imputation.

```python
# Without coerce — crashes on first bad row
df['revenue'] = df['revenue'].astype(float)  # ValueError if any row has '$1,234'

# With coerce — converts valid rows, turns bad rows into NaN
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')
```

### Dtype Validation Checklist

After all type corrections, run a final validation:

```python
expected = {
    'order_date': 'datetime64[ns]',
    'age': 'float64',
    'revenue': 'float64',
    'quantity': 'float64',
    'gender': 'category',
    'order_status': 'category',
}
for col, expected_type in expected.items():
    actual = str(df[col].dtype)
    status = 'PASS' if actual == expected_type else 'FAIL'
    print(f'{status} | {col}: {actual} (expected: {expected_type})')
```


In [ ]:
# order_date → datetime64 (already done in Task 5)assert df['order_date'].dtype == 'datetime64[ns]', "order_date should be datetime64"print(f"order_date dtype: {df['order_date'].dtype} ")# customer_satisfaction → nullable Int64df['customer_satisfaction'] = df['customer_satisfaction'].astype('Int64')print(f"customer_satisfaction dtype: {df['customer_satisfaction'].dtype} ")# return_flag → bool# Handle string 'True'/'False' and actual booldf['return_flag'] = df['return_flag'].map({True:True, False:False, 'True':True, 'False':False, 1:True, 0:False}).astype(bool)print(f"return_flag dtype: {df['return_flag'].dtype} ")# quantity, days_to_ship → intdf['quantity'] = df['quantity'].round().astype(int)df['days_to_ship'] = df['days_to_ship'].round().astype(int)print(f"quantity dtype: {df['quantity'].dtype} ")print(f"days_to_ship dtype: {df['days_to_ship'].dtype} ")# High-cardinality strings → Categorical (memory optimization)mem_before = df.memory_usage(deep=True).sum() / 1024for col in ['product_category','region','gender','payment_method','order_status']:df[col] = df[col].astype('category')mem_after = df.memory_usage(deep=True).sum() / 1024print(f"\nMemory before Categorical: {mem_before:,.1f} KB")print(f"Memory after Categorical: {mem_after:,.1f} KB")print(f"Reduction: {(1 - mem_after/mem_before)*100:.1f}%")

---
## Task 8 — Final Validation and Export

### Theory: The Data Contract

A data contract is a formal specification of what a clean dataset must satisfy. It defines rules that every row must meet before the data is considered production-ready. Implementing a data contract as executable code turns quality checks into automated tests.

### Why Automate Validation?

In production data pipelines, the same cleaning code runs on new data every day. If upstream data quality degrades (a new system introduces a new date format, a vendor starts sending negative prices), the pipeline must detect and alert — not silently produce wrong results.

Python's `assert` statement is the simplest way to implement data contracts:

```python
assert condition, "Error message if condition fails"
```

If the condition is False, execution stops and the error message is displayed. In production, replace `assert` with proper exception handling and alerting.

### Validation Checklist

| Check | Code |
|---|---|
| No missing values remain | `assert df.isnull().sum().sum() == 0` |
| No duplicate rows | `assert df.duplicated().sum() == 0` |
| Age is in valid range | `assert df['age'].between(0, 120).all()` |
| Revenue is non-negative | `assert (df['revenue'] >= 0).all()` |
| Quantity is positive | `assert (df['quantity'] > 0).all()` |
| Order date is within expected range | `assert df['order_date'].between('2020-01-01', '2025-12-31').all()` |
| Expected columns all present | `assert set(required_cols).issubset(df.columns)` |
| Row count is within expected range | `assert 4000 <= len(df) <= 4400` |

### Documenting Cleaning Decisions

Every cleaning decision should be logged. A cleaning summary should record:
- Rows before cleaning vs after
- Missing values by column: before vs after
- Duplicate rows removed
- Outlier treatment applied per column
- Any columns added or removed

This documentation is your audit trail and is essential for reproducibility.

### Saving the Clean Dataset

Always save to a *new file* — never overwrite raw data:

```python
df.to_csv('retail_sales_clean.csv', index=False, encoding='utf-8')
```

Use `index=False` to prevent pandas from saving the row index as an extra column.


In [ ]:
def validate_dataframe(df):"""Run data quality checks. Returns a pass/fail report DataFrame."""checks = []def chk(name, mask_fail, note=''):n = mask_fail.sum() if hasattr(mask_fail,'sum') else int(not mask_fail)checks.append({'Check': name, 'Failing Rows': n,'Status': 'PASS ' if n == 0 else 'FAIL ', 'Note': note})# 1. No nulls in critical columnsfor col in ['order_id','order_date','region','product_category','sales_amount']:chk(f'No nulls in {col}', df[col].isnull())# 2. Age in valid rangechk('age ∈ [18, 90]', (df['age'] < 18) | (df['age'] > 90))# 3. Quantity positivechk('quantity > 0', df['quantity'] <= 0)# 4. Gender validchk('gender ∈ {Male,Female,Other}',~df['gender'].isin(['Male','Female','Other']),'Unexpected gender values')# 5. Region validchk('region ∈ 5 valid values',~df['region'].isin(['North','South','East','West','Central']),'Unexpected region values')# 6. No future dateschk('No dates after 2024-12-31', df['order_date'] > pd.Timestamp('2024-12-31'))# 7. sales_amount positivechk('sales_amount > 0', df['sales_amount'] <= 0)return pd.DataFrame(checks)report = validate_dataframe(df)print(report.to_string(index=False))print(f"\nOverall: {'ALL CHECKS PASSED ' if (report['Failing Rows']==0).all() else 'SOME CHECKS FAILED '}")

In [ ]:
# Cleaning summaryprint("=" * 55)print(" DATA CLEANING SUMMARY")print("=" * 55)print(f" Original shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")print(f" Cleaned shape : {df.shape[0]:,} rows × {df.shape[1]} columns")print(f" Rows removed : {df_raw.shape[0] - df.shape[0]:,}")print(f" Columns added : {df.shape[1] - df_raw.shape[1]} (flag + time features)")print(f" Missing cells : {df.isnull().sum().sum()} (was {df_raw.isnull().sum().sum():,})")print("=" * 55)# Export clean datasetdf.to_csv('retail_sales_clean.csv', index=False)print("\n Saved: retail_sales_clean.csv")

---
## Assignment 1 — Data Cleaning — Complete! Well done! Proceed to the next assignment notebook. | Assignment | Notebook File |
|---|---|
| 1. Data Cleaning | `A1_Data_Cleaning.ipynb` |
| 2. EDA | `A2_EDA.ipynb` |
| 3. Statistical Analysis | `A3_Statistical_Analysis.ipynb` |
| 4. Segmentation & Clustering | `A4_Segmentation_Clustering.ipynb` |
| 5. Time Series Forecasting | `A5_Time_Series_Forecasting.ipynb` |